# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their @id
print("Record Sets Available:")
record_sets = {rs['@id']: rs for rs in meta.to_json().get('recordSet', [])}
for rs_id, rs_obj in record_sets.items():
    print(f"  RecordSet @id: {rs_id}  Name: {rs_obj.get('name', 'N/A')}")

# For each record set, list its fields
for rs_id, rs_obj in record_sets.items():
    print(f"\nFields in RecordSet {rs_id}:")
    for field in rs_obj.get('field', []):
        field_id = field.get('@id', field) if isinstance(field, dict) else field
        print(f"  Field @id: {field_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll try to extract all dataframes for all record sets present in the schema
dataframes = {}
record_set_ids = list(record_sets.keys())

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load RecordSet {record_set_id}: {e}")

# For demonstration, work with the first non-empty dataframe
main_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rs_id
        break

if main_record_set_id:
    print(f"Selected RecordSet for further analysis: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record set with tabular data was found!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    
    # Show numeric columns
    numeric_columns = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
    print(f"Numeric columns available: {numeric_columns}")
    
    if len(numeric_columns) > 0:
        numeric_field = numeric_columns[0]
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as an example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by any non-numeric field, e.g., the first string/categorical field
        potential_group_fields = df.select_dtypes(include=['object']).columns.tolist()
        # Avoid grouping by all-string fields like description; try to find something meaningful
        group_field = None
        for col in potential_group_fields:
            n_unique = df[col].nunique()
            if 1 < n_unique < len(df) * 0.5:  # e.g., not all-unique or only-constant
                group_field = col
                break
        
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} and mean {numeric_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field to demonstrate grouping.")
    else:
        print("No numeric columns found in DataFrame for EDA.")
else:
    print("No record set loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Check for numeric columns again
    numeric_columns = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
    if len(numeric_columns) > 0:
        field = numeric_columns[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(df[field].dropna(), kde=True)
        plt.xlabel(field)
        plt.title(f'Distribution of {field}')
        plt.tight_layout()
        plt.show()
        # If we have a categorical field, create a boxplot
        group_fields = df.select_dtypes(include=['object']).columns.tolist()
        if group_fields:
            for group_field in group_fields:
                unique_vals = df[group_field].nunique()
                if 1 < unique_vals < 50:
                    plt.figure(figsize=(10, 5))
                    sns.boxplot(x=group_field, y=field, data=df)
                    plt.title(f'Boxplot of {field} by {group_field}')
                    plt.xticks(rotation=45, ha='right')
                    plt.tight_layout()
                    plt.show()
                    break
    else:
        print("No numeric columns for visualization.")
else:
    print("No record set loaded for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to use the `mlcroissant` library to load and explore the FAIR2 mass spectrometry dataset. We:
* Loaded dataset metadata to get a high-level summary.
* Inspected available record sets and fields using their `@id` references.
* Loaded tabular data for analysis, filtered on numeric fields, and normalized results to prepare for downstream applications.
* Visualized feature distributions and explored groupings to uncover structure in the data.

These steps establish a reusable workflow for FAIR-compliant datasets described using Croissant schemas. For deeper scientific or domain analysis, consult the relevant field dictionaries and documentation included in the dataset.